In [3]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

C:\Users\Dell\AppData\Local\Temp\ipykernel_3500\659718525.py:3: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  str   
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(1), str(1)
memory usage: 69.8+ MB


In [5]:
df['Date'] = pd.to_datetime(df['Date'])

In [6]:
df = df[df['Open'] == 1]

In [7]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [8]:
store_sales = df.groupby(['Store', 'Date'])['Sales'].sum().reset_index()

In [9]:
import numpy as np

def get_trend(group):
    x = np.arange(len(group))
    y = group['Sales'].values
    if len(y) < 2:
        return 0
    return np.polyfit(x, y, 1)[0]  # slope

trends = store_sales.groupby('Store').apply(get_trend).reset_index()
trends.columns = ['Store', 'Trend']

In [10]:
declining_stores = trends.sort_values(by='Trend').head(10)
declining_stores

,Store,Trend
522,523,-7.720553
1085,1086,-6.142672
814,815,-6.123946
904,905,-6.109592
982,983,-5.471419
816,817,-5.024319
985,986,-4.591905
1111,1112,-4.364882
684,685,-4.038112
56,57,-3.711678


# Declining Sales Identification

Sales trends were calculated for each store using a linear trend approach. Stores with the most negative slope values were identified as having declining sales. The results show that stores such as 523, 1086, and 815 exhibit the strongest downward trends in sales over time.

In [11]:
store_523 = store_sales[store_sales['Store'] == 523]
store_1086 = store_sales[store_sales['Store'] == 1086]

In [19]:
promo_effect = df.groupby('Promo')['Sales'].mean()
promo_effect

Promo
0    5929.407603
1    8228.281239
Name: Sales, dtype: float64

In [20]:
lift = (promo_effect[1] - promo_effect[0]) / promo_effect[0]
lift

np.float64(0.3877071352874008)

In [21]:
def simulate_markdown(df, lift):
    df = df.copy()
    df['Sales_10pct'] = df['Sales'] * (1 + lift * 0.5)
    df['Sales_20pct'] = df['Sales'] * (1 + lift)
    return df

store_523_sim = simulate_markdown(store_523, lift)
store_1086_sim = simulate_markdown(store_1086, lift)

Markdown impact was estimated using the observed effect of promotions in the dataset. The average increase in sales during promotional periods was used as a proxy for price sensitivity. This data-driven lift was applied to simulate the impact of 10% and 20% markdown scenarios.

In [13]:
store_523_sim['Month'] = store_523_sim['Date'].dt.month
store_1086_sim['Month'] = store_1086_sim['Date'].dt.month

In [17]:
store_523_sim.groupby('Month')['Sales'].mean()


Month
1     15213.217949
2     14981.361111
3     16002.558442
4     15762.849315
5     15662.222222
6     15522.586667
7     14943.555556
8     15534.943396
9     15252.000000
10    14805.365385
11    15008.823529
12    18142.571429
Name: Sales, dtype: float64

In [18]:
store_1086_sim.groupby('Month')['Sales'].mean()

Month
1     7338.858974
2     7361.847222
3     7757.363636
4     7907.041096
5     7882.380282
6     7644.178082
7     7685.654321
8     7614.377358
9     7231.980392
10    7439.673077
11    7727.960784
12    8682.612245
Name: Sales, dtype: float64

# Seasonality Analysis

Both stores show peak sales in December, indicating strong seasonal demand. Lower sales are observed in specific months such as early-year periods (January–February) and certain mid-year months.

# Markdown Timing Strategy

Store 523:

Markdown should be applied during low-sales months such as February, July, and October to boost demand. Discounts should be avoided in December due to already high sales.

Store 1086:

Markdown is most effective during low-demand periods such as January, February, and September. Similar to Store 523, December does not require discounts due to peak sales.